Notebook 2 – SQL Enterprise Data Layer

This notebook builds a production-style star schema to support:
  KPI tracking
  Revenue segmentation
  Churn analysis
  Customer value modeling

Designed to mirror Corporate FP&A reporting architecture.

In [ ]:
import pandas as pd
import sqlite3

In [ ]:
# Load dataset
df = pd.read_csv("/content/Telco-Customer-Churn.csv")

# Clean TotalCharges
df["TotalCharges"] = df["TotalCharges"].replace(" ", pd.NA)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"])
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# Create churn flag
df["churn_flag"] = df["Churn"].map({"Yes": 1, "No": 0})

# Create pricing tier
def assign_plan(monthly_charge):
    if monthly_charge < 40:
        return "Basic"
    elif monthly_charge < 90:
        return "Pro"
    else:
        return "Enterprise"

df["plan_tier"] = df["MonthlyCharges"].apply(assign_plan)

In [ ]:
conn = sqlite3.connect("finance_engine.db")
cursor = conn.cursor()

print("Database connection established.")

Database connection established.


In [ ]:
# Customer Dimension Table
cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_customer (
    customer_id TEXT PRIMARY KEY,
    gender TEXT,
    senior_citizen INTEGER,
    partner TEXT,
    dependents TEXT,
    tenure INTEGER,
    plan_tier TEXT
)
""")

# Subscription Fact Table
cursor.execute("""
CREATE TABLE IF NOT EXISTS fact_subscription (
    customer_id TEXT,
    monthly_charges REAL,
    total_charges REAL,
    churn_flag INTEGER,
    FOREIGN KEY(customer_id) REFERENCES dim_customer(customer_id)
)
""")

print("Star schema created successfully.")



Star schema created successfully.


In [ ]:
# Insert into dimension table
customer_records = df[[
    "customerID",
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "tenure",
    "plan_tier"
]].values.tolist()

cursor.executemany("""
INSERT OR REPLACE INTO dim_customer
(customer_id, gender, senior_citizen, partner, dependents, tenure, plan_tier)
VALUES (?, ?, ?, ?, ?, ?, ?)
""", customer_records)

# Insert into fact table
subscription_records = df[[
    "customerID",
    "MonthlyCharges",
    "TotalCharges",
    "churn_flag"
]].values.tolist()

cursor.executemany("""
INSERT OR REPLACE INTO fact_subscription
(customer_id, monthly_charges, total_charges, churn_flag)
VALUES (?, ?, ?, ?)
""", subscription_records)

conn.commit()

print("Data successfully loaded into warehouse.")


Data successfully loaded into warehouse.


In [ ]:
# ---- Overall Business KPIs ----
overall_query = """
SELECT
    COUNT(*) AS total_customers,
    SUM(monthly_charges) AS total_mrr,
    SUM(churn_flag) AS churned_customers,
    SUM(churn_flag) * 1.0 / COUNT(*) AS churn_rate
FROM fact_subscription
"""

overall_kpis = cursor.execute(overall_query).fetchone()

overall_df = pd.DataFrame(
    [overall_kpis],
    columns=["Total Customers", "Total MRR", "Churned Customers", "Churn Rate"]
)

print("\nOverall Business KPIs")
print(overall_df)


# ---- Plan-Level Revenue & Churn ----
plan_query = """
SELECT
    c.plan_tier,
    COUNT(*) AS customers,
    SUM(f.monthly_charges) AS total_mrr,
    AVG(f.monthly_charges) AS avg_mrr,
    SUM(f.churn_flag) * 1.0 / COUNT(*) AS churn_rate
FROM dim_customer c
JOIN fact_subscription f
ON c.customer_id = f.customer_id
GROUP BY c.plan_tier
ORDER BY total_mrr DESC
"""

plan_summary = cursor.execute(plan_query).fetchall()

plan_df = pd.DataFrame(
    plan_summary,
    columns=["Plan", "Customers", "Total MRR", "Average MRR", "Churn Rate"]
)

print("\nPlan-Level Performance")
print(plan_df)


# ---- Revenue at Risk ----
risk_query = """
SELECT
    c.plan_tier,
    SUM(f.monthly_charges * f.churn_flag) AS revenue_at_risk
FROM dim_customer c
JOIN fact_subscription f
ON c.customer_id = f.customer_id
GROUP BY c.plan_tier
ORDER BY revenue_at_risk DESC
"""

revenue_risk = cursor.execute(risk_query).fetchall()

risk_df = pd.DataFrame(
    revenue_risk,
    columns=["Plan", "Revenue at Risk"]
)

print("\nRevenue at Risk by Plan")
print(risk_df)


# ---- Average Customer Value ----
value_query = """
SELECT
    c.plan_tier,
    AVG(f.monthly_charges * c.tenure) AS avg_customer_value
FROM dim_customer c
JOIN fact_subscription f
ON c.customer_id = f.customer_id
GROUP BY c.plan_tier
"""

customer_value = cursor.execute(value_query).fetchall()

value_df = pd.DataFrame(
    customer_value,
    columns=["Plan", "Average Customer Value"]
)

print("\nAverage Customer Value by Plan")
print(value_df)


Overall Business KPIs
   Total Customers  Total MRR  Churned Customers  Churn Rate
0            21129  1368349.8               5607     0.26537

Plan-Level Performance
         Plan  Customers  Total MRR  Average MRR  Churn Rate
0         Pro      10386  713970.75    68.743573    0.312825
1  Enterprise       5232  528364.50   100.987099    0.328555
2       Basic       5511  126014.55    22.866004    0.115950

Revenue at Risk by Plan
         Plan  Revenue at Risk
0         Pro        230200.65
1  Enterprise        171162.30
2       Basic         16029.60

Average Customer Value by Plan
         Plan  Average Customer Value
0       Basic              661.735275
1  Enterprise             4588.718148
2         Pro             1974.800491


In [ ]:
print("\nData Integrity Check")
print("dim_customer count:",
      cursor.execute("SELECT COUNT(*) FROM dim_customer").fetchone()[0])

print("fact_subscription count:",
      cursor.execute("SELECT COUNT(*) FROM fact_subscription").fetchone()[0])


Data Integrity Check
dim_customer count: 7043
fact_subscription count: 21129


In [ ]:
conn.close()
print("\nDatabase connection closed.")



Database connection closed.
